In [1]:
import torch

# you might have to install this (minicons), we dont need to use it per se, 
# but it has good functionality for sequence scoring
from minicons import scorer 

from torch import optim
from tqdm import trange, tqdm
from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup, get_constant_schedule, set_seed
from PIL import Image

In [ ]:
from transformers import set_seed
set_seed(42)
lm = scorer.VLMScorer("Qwen/Qwen3-VL-4B-Instruct", 
                      device="cuda", 
                      torch_dtype=torch.bfloat16, 
                    )

def chat_template(self, text, noimage=False, assistant=False):
    if noimage == False:
        context = [
            {
                "role": "user",
                "content": [{"type": "image"},{"type": "text", "text": text}],
            }
        ]
    else:
        context = [
            {
                "role": "user",
                "content": [{"type": "text", "text": text}],
            }
        ]

    if not assistant:
        context = self.tokenizer.apply_chat_template(
            context, continue_final_message=True
        )
    else:
        context = self.tokenizer.apply_chat_template(
            context, add_generation_prompt=True
        )
    return context


# Default prompt is to caption the image -- we could play around with this as well, like have it
# be something like "Answer the question", and train on wug and wugs in a QA format, tons of things
# to think about!

def train_chat_template(self, text):
    context = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Caption this image."},
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": text}
            ]
        }
    ]
    return self.tokenizer.apply_chat_template(context, add_generation_prompt=False).strip()


VISUAL_PATH_BEST="../embeddings/Qwen3-VL-4B/vision/qwen3_vl_4b_vision.pt"
SYNTAX_PATH_BEST= "embeddings/Qwen3-VL-4B/vision/qwen3_vl_4b_syntax.pt"

added_tokens = [" [wug]", " [wugs]"]
existing_vocab = lm.tokenizer.tokenizer.get_vocab()
tokens_to_add = [t for t in added_tokens if t not in existing_vocab]
if tokens_to_add:
    lm.tokenizer.tokenizer.add_tokens(tokens_to_add)
    old_len = lm.model.resize_token_embeddings().weight.shape[0]
    lm.model.resize_token_embeddings(old_len + len(tokens_to_add))

saved = torch.load(VISUAL_PATH_BEST)
# saved = torch.load(SYNTAX_PATH_BEST)

emb = lm.model.language_model.embed_tokens
emb.weight.data[saved["wug_id"]] = saved["wug_embedding"].to(emb.weight.device, dtype=emb.weight.dtype)
emb.weight.data[saved["wugs_id"]] = saved["wugs_embedding"].to(emb.weight.device, dtype=emb.weight.dtype)

print(f"Loaded embeddings for [wug] (id={saved['wug_id']}) and [wugs] (id={saved['wugs_id']})")
print(f"  [wug]  norm: {emb.weight[saved['wug_id']].norm().item():.4f}")
print(f"  [wugs] norm: {emb.weight[saved['wugs_id']].norm().item():.4f}")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 